# Supabase Database - Tabelas (CRUD)

Documentação da integração do BetterAI com as tabelas do Supabase: como obter a integração, declarar tabelas e executar CRUD, consultas, paginação e operações em lote.

### Visão Geral

O acesso às tabelas do Supabase é feito por uma única integração, localizada em `src/database/supabase_integration/`. Os fluxos do projeto não criam clientes do SDK `supabase` nem escrevem SQL: pedem a integração ao ponto de composição e operam sobre repositórios de tabela.

- **`get_supabase()`** (`src/database/supabase_provider.py`): devolve a integração compartilhada do processo, criada na primeira chamada a partir das variáveis `SBX_*`.
- **`TableDescriptor`**: descreve uma tabela (nome, chave primária, colunas permitidas, colunas gerenciadas pelo servidor, modelo).
- **`sb.tables.table("nome")`**: devolve o `TableRepository` com as operações de CRUD, consulta, paginação e lote.
- **`Query`**, **`Condition`**, **`FilterGroup`**, **`Sort`**, **`PageRequest`**: objetos de consulta independentes do SDK.

> A integração fala com o Supabase pela API REST (PostgREST). Não há execução de SQL cru: filtros, ordenações e escritas são descritos com os objetos de consulta e validados antes de ir à rede.

### Fluxo de Execução

O fluxo de uma operação de tabela segue estes passos:

1. **Configuração:** `get_supabase()` lê as variáveis `SBX_*`, valida tudo de uma vez e falha cedo, listando só os nomes ausentes ou inválidos.
2. **Registro:** as tabelas declaradas em `_table_descriptors()` são registradas na criação da integração.
3. **Repositório:** `sb.tables.table("nome")` devolve o repositório da tabela, desde que ela esteja registrada e permitida.
4. **Validação:** colunas, filtros e valores são validados antes da rede (`ValidationError`, `MappingError`).
5. **Execução:** a chamada passa pelo executor de resiliência (timeout de orçamento e retry com backoff, só em operações idempotentes).
6. **Resultado:** a linha, a lista, a página ou o resultado de lote volta como `dict` (ou como o `model` do descritor); qualquer falha vira uma exceção do domínio (`SupabaseIntegrationError` e subclasses).

### Configuração

A configuração vem das variáveis de ambiente `SBX_*`. As usadas pelas tabelas são:

| Variável | Descrição |
|---|---|
| `SBX_URL` | URL https do projeto Supabase (obrigatória) |
| `SBX_SECRET_KEY` | Chave secret do backend (obrigatória; nunca no frontend) |
| `SBX_TABLES_ALLOWLIST` | Tabelas permitidas, separadas por vírgula (obrigatória em prod) |
| `SBX_DB_SCHEMA` | Schema Postgres (padrão `public`) |
| `SBX_TABLES_DEFAULT_PAGE_SIZE` | Tamanho de página padrão (50) |
| `SBX_TABLES_MAX_PAGE_SIZE` | Teto do tamanho de página (padrão `min(1000, server max rows)`) |
| `SBX_TABLES_BATCH_SIZE` | Tamanho de cada chunk nas operações em lote (500) |
| `SBX_TABLES_SERVER_MAX_ROWS` | `max-rows` real do projeto Supabase (1000) |
| `SBX_TABLES_QUERY_TIMEOUT_SECONDS` | Timeout das consultas em segundos (10) |
| `SBX_TABLES_BATCH_TIMEOUT_SECONDS` | Timeout das operações em lote em segundos (60) |

> **Regras de ambiente.** Em `homol` e `prod` o arquivo `.env` é proibido (use as variáveis da plataforma) e `prod` exige `SBX_TABLES_ALLOWLIST`. Uma tabela nova precisa de um `TableDescriptor` **e** estar na allowlist. A lista completa está em `src/database/supabase_integration/.env.example`.

### Obtendo a Integração

```python
from src.database.supabase_provider import get_supabase

sb = get_supabase()  # Integração compartilhada do processo
items = sb.tables.table("sbx_test_items")  # Repositório da tabela
```

As tabelas usadas pelos fluxos são declaradas em `_table_descriptors()`, em `src/database/supabase_provider.py`. Adicionar uma tabela é adicionar um descritor, sem editar mais nenhum código existente:

```python
from src.database.supabase_integration import TableDescriptor

def _table_descriptors():
    return (
        TableDescriptor(
            "sbx_test_items",
            columns=("id", "name", "qty", "created_at"),  # Allowlist de colunas
            server_managed_columns=("id", "created_at"),  # Preenchidas pelo servidor
            sort_key="created_at",  # Habilita paginação por keyset
        ),
    )
```

### TableDescriptor

| Campo | Significado |
|---|---|
| `name` | Nome da tabela. Precisa estar em `SBX_TABLES_ALLOWLIST` se ela existir. |
| `schema` | Schema Postgres (padrão `public`). |
| `primary_key` | Coluna da chave primária, usada por `get_by_id`, `update`, `delete` e no desempate da ordenação (padrão `id`). |
| `model` | `None` devolve `dict`; também aceita `dataclass` ou classe estilo Pydantic. |
| `server_managed_columns` | Colunas que o servidor preenche (`id`, `created_at`). Não são enviadas no insert quando `None` e são recusadas no `update`. |
| `columns` | Allowlist de colunas para filtro, ordenação, escrita e seleção. `None` não restringe. Precisa conter a chave primária e a `sort_key`. |
| `sort_key` | Coluna estável e não nula que habilita a paginação por keyset (ex.: `created_at`). |

### Create

```python
novo = items.insert({"name": "Ana", "qty": 1})  # Devolve a linha gravada, já com id e created_at
print(novo["id"])

lote = items.insert_many(  # Lote: um resultado por item
    [{"name": f"item-{i}", "qty": i} for i in range(25)],
    BatchOptions(batch_size=10, mode="continue"),
)
print(lote.total, lote.succeeded, lote.failed, lote.all_ok, lote.stopped_early)
for it in lote.items:
    if not it.ok:
        print("failed item", it.index, type(it.error).__name__)
```

### Read

```python
um = items.get_by_id(novo["id"])  # RecordNotFoundError se não existe

achados = items.find(Query.where(Condition("name", Op.ILIKE, "an%")))  # Lista vazia se nada casa
iguais = items.find(Query.eq("qty", 5))

q = Query(
    filter=FilterGroup(items=(
        Condition("qty", Op.GTE, 5),
        FilterGroup(items=(Condition("qty", Op.EQ, 7), Condition("qty", Op.EQ, 9)), logic="or"),
    )),  # qty >= 5 AND (qty = 7 OR qty = 9)
    sorts=(Sort("name"),),
    columns=("id", "name", "qty"),  # Projeção opcional
)
pag = items.list(q, PageRequest(limit=10, offset=0, include_count=True))
print(pag.items, pag.total, pag.has_more)

for linha in items.iterate(q, page_size=100):  # Varredura de tabela grande, uma página por vez
    ...
```

> `find()` carrega todas as linhas que casam em memória. Para tabelas grandes use `iterate()`.

### Update

```python
atual = items.update(novo["id"], {"qty": 5})  # RecordNotFoundError se não existe
n = items.update_where(Condition("name", Op.LIKE, "item-%"), {"qty": 0})  # Devolve o número de linhas afetadas
items.upsert({"id": novo["id"], "name": "Ana", "qty": 999}, on_conflict="id")  # Insere ou atualiza
items.upsert_many([{"id": 1, "name": "A"}, {"id": 2, "name": "B"}], on_conflict="id")
```

### Delete

```python
r = items.delete(novo["id"])  # Idempotente
print(r.removed, r.removed_count)  # removed=False se já não existia
items.delete_where(Condition("name", Op.LIKE, "item-%"))  # Filtro vazio é recusado (ValidationError)
```

> `update_where` e `delete_where` recusam filtro vazio (`ValidationError`). É a proteção contra alterar ou apagar a tabela inteira.

### Consultas

`Query` tem `filter`, `sorts` e `columns`. Atalhos: `Query.where(*conditions)` (AND entre as condições) e `Query.eq(coluna, valor)`. Um `FilterGroup` combina `Condition` ou outros grupos com `logic="and"` ou `"or"` (profundidade máxima 8).

| Operador | Significado | Regra do valor |
|---|---|---|
| `EQ`, `NEQ`, `GT`, `GTE`, `LT`, `LTE` | Comparações | Escalar não nulo (use `IS_NULL`/`NOT_NULL` para nulos) |
| `IN` | Pertence a | Lista, tupla ou conjunto, até 1000 itens |
| `LIKE`, `ILIKE` | Padrão com `%` | Texto |
| `IS_NULL`, `NOT_NULL` | Nulo / não nulo | O valor é ignorado |
| `CONTAINS` | `@>` em jsonb ou array | Lista, dict ou texto |

Valores aceitos nas escritas: `str`, `int`, `bool`, `float` finito, `Decimal` (enviado como texto), `datetime` **com fuso** (convertido para UTC), `date`, `time`, `UUID`, `Enum`, `dict`, `list`, `tuple` e `set`. Qualquer outro tipo levanta `MappingError`.

### Paginação

`PageRequest(limit=None, offset=0, cursor=None, include_count=False)` pede uma página e `Page` devolve `items`, `limit`, `offset`, `next_cursor`, `total` e `has_more`. Sem `limit`, vale `SBX_TABLES_DEFAULT_PAGE_SIZE`; pedir mais que o teto efetivo reduz o valor com aviso, sem erro.

|  | Offset (padrão) | Keyset (opcional) |
|---|---|---|
| Quando | Páginas rasas ou saltar para a página N | Varrer tabelas grandes e páginas profundas |
| Exige | Nada | `sort_key` no descritor e ordenação compatível |
| Custo | Cresce com o offset | Constante |
| Estabilidade | Pode repetir ou pular linhas se os dados mudam | Estável |

```python
from src.database.supabase_integration.tables import KEYSET_START

cursor = KEYSET_START  # Abre a cadeia de páginas
while cursor:
    pag = items.list(Query(), PageRequest(limit=100, cursor=cursor))
    for linha in pag.items:
        ...
    cursor = pag.next_cursor  # None na última página
```

> O cursor é opaco e assinado pela consulta: usá-lo com outro filtro ou direção de ordenação levanta `ValidationError`.

### Operações em Lote

`insert_many` e `upsert_many` recebem `BatchOptions(batch_size=None, mode="continue")`. Cada chunk é uma requisição atômica no Postgres; não há atomicidade entre chunks. O lote inteiro é validado antes de qualquer envio.

- `mode="continue"`: um chunk que falha por dado ou conflito é dividido ao meio até isolar os itens ruins; o restante é gravado.
- `mode="stop"`: para no primeiro chunk com falha e marca `stopped_early=True` se sobrou lote.
- `BatchResult` tem `items` (lista de `ItemOutcome` com `index`, `ok`, `entity`, `error`), `stopped_early` e as propriedades `total`, `succeeded`, `failed` e `all_ok`.

### Entidades Tipadas

Por padrão as linhas são `dict`. Para trabalhar com `dataclass` ou classe estilo Pydantic, informe `model` no descritor. `insert` e `upsert` aceitam o modelo ou `dict`; as leituras devolvem o modelo; `update(id, {...})` recebe sempre um `dict` de alterações.

```python
from dataclasses import dataclass
from datetime import datetime
from typing import Optional
from uuid import UUID

@dataclass
class Item:
    id: UUID
    name: str
    qty: int = 0
    created_at: Optional[datetime] = None

TableDescriptor("sbx_test_items", model=Item, server_managed_columns=("id", "created_at"))

novo = items.insert(Item(id=None, name="Ana", qty=1))  # id=None é omitido (coluna do servidor)
outro = items.get_by_id(novo.id)  # Leituras devolvem Item
```

### Tabela de Métodos

| Método | Retorno | Descrição Breve | Retry automático |
|---|---|---|---|
| `get_by_id(id)` | entidade | Busca por chave primária; `RecordNotFoundError` se não existe. | Sim |
| `find(query=None)` | `list` | Todas as linhas que casam, em memória; `[]` se nada casa. | Sim |
| `list(query=None, page=None)` | `Page` | Uma página, por offset ou keyset. | Sim |
| `iterate(query=None, page_size=None)` | iterador | Varredura preguiçosa, uma página por vez. | Sim |
| `insert(entity)` | entidade | Grava uma linha e devolve a linha gravada. | Não |
| `insert_many(entities, options=None)` | `BatchResult` | Grava em lote, com resultado por item. | Não |
| `update(id, changes)` | entidade | Atualiza por chave primária; `RecordNotFoundError` se não existe. | Sim |
| `update_where(filter, changes)` | `int` | Atualiza por filtro e devolve as linhas afetadas; filtro vazio é recusado. | Sim |
| `upsert(entity, on_conflict=None)` | entidade | Insere ou atualiza; `on_conflict` padrão é a chave primária. | Sim |
| `upsert_many(entities, on_conflict=None, options=None)` | `BatchResult` | Upsert em lote. | Sim |
| `delete(id)` | `DeleteResult` | Remove por chave primária; idempotente. | Sim |
| `delete_where(filter)` | `DeleteResult` | Remove por filtro; filtro vazio é recusado. | Sim |

### Tratamento de Erros

Nenhuma exceção do SDK (`supabase`, `postgrest`, `httpx`) chega ao código do projeto: tudo vira `SupabaseIntegrationError` ou uma subclasse, com a causa original em `__cause__`.

| Exceção | Quando |
|---|---|
| `ConfigurationError` | Tabela não registrada ou fora da allowlist; tabela ou coluna inexistente no banco. |
| `ValidationError` | Entrada inválida, detectada antes da rede (coluna fora da allowlist, filtro vazio, valor inválido). |
| `MappingError` | Falha ao converter valor ou linha; cita tabela e campo, nunca o valor. |
| `RecordNotFoundError` | `get_by_id` ou `update` sem linha correspondente. |
| `DuplicateRecordError`, `ReferenceViolationError` | Violação de unicidade (`23505`) ou de chave estrangeira (`23503`). |
| `DataError` | Tipo, not-null ou check rejeitados pelo banco. |
| `PermissionDeniedError`, `AuthenticationError` | Política de RLS negou o acesso; chave ou JWT inválido. |
| `OperationTimeoutError`, `ServiceUnavailableError`, `RateLimitError` | Falhas transitórias (`retryable=True`). |

```python
try:
    items.get_by_id("00000000-0000-0000-0000-000000000000")
except RecordNotFoundError:
    ...  # 404 no endpoint
except ValidationError:
    ...  # 422: entrada inválida
except SupabaseIntegrationError as e:  # Qualquer erro da integração
    print(e.retryable, e.status, e.provider_code, e.attempts, e.outcome_unknown, e.correlation_id)
```

> `insert` e `insert_many` nunca são repetidos automaticamente (não são idempotentes). Em timeout o erro traz `outcome_unknown=True`: confira no banco antes de tentar de novo.

### Arquitetura e Insights

- **Lista de permissões:** a tabela precisa estar registrada e em `SBX_TABLES_ALLOWLIST`; as colunas de filtro, ordenação e escrita podem ser restritas por `columns`.
- **Segredos protegidos:** a chave `SBX_SECRET_KEY` é um `Secret` que nunca aparece em `repr`, logs ou mensagens de erro.
- **Resiliência:** só operações idempotentes têm retry com backoff e jitter, dentro de um orçamento de tempo (`SBX_RETRY_*`).
- **Sem bloquear o event loop:** em FastAPI use `create_async_integration`, que executa as chamadas em threads.
- **Arquivo e registro juntos:** para gravar um arquivo e o registro que aponta para ele, com compensação em caso de falha, veja a página de Storage.